# Module 10: Machine Learning with Scikit-Learn
# Lab 01: The ML Workflow & Feature Engineering

**Prepared by Information Tech Consultants Ltd**

---

## 🎯 Learning Objectives
By the end of this notebook, you will be able to:
- [ ] Describe the end-to-end machine learning workflow
- [ ] Load and split data into training and test sets
- [ ] Apply common feature engineering techniques (encoding, scaling, imputation)
- [ ] Select relevant features using built-in scikit-learn tools

**⏱ Estimated Time:** 75 minutes  
**📋 Prerequisites:** Modules 1–9 (Python fundamentals, Pandas, NumPy, EDA, Statistics)

In [ ]:
# ============================================================
# 📦 Environment Setup — Run this cell first!
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# We'll use scikit-learn throughout this module
import sklearn
print(f"✅ All packages imported successfully!")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

In [ ]:
# ============================================================
# 🛠 Helper Functions (run once, use throughout)
# ============================================================

from IPython.display import HTML, display

def info_box(title, content, color="#0092D6", bg="#E3F2FD"):
    """Display a styled information callout box."""
    display(HTML(f"""
    <div style="background:{bg};padding:15px;border-left:5px solid {color};
    margin:10px 0;border-radius:4px;font-family:Calibri,Arial,sans-serif;">
    <strong>💡 {title}</strong><br>{content}</div>"""))

def warning_box(title, content):
    """Display a warning callout box."""
    display(HTML(f"""
    <div style="background:#FFF3E0;padding:15px;border-left:5px solid #FF9800;
    margin:10px 0;border-radius:4px;font-family:Calibri,Arial,sans-serif;">
    <strong>⚠️ {title}</strong><br>{content}</div>"""))

def interview_box(question, key_points):
    """Display an interview question callout box."""
    display(HTML(f"""
    <div style="background:#F3E5F5;padding:15px;border-left:5px solid #9C27B0;
    margin:10px 0;border-radius:4px;font-family:Calibri,Arial,sans-serif;">
    <strong>🎯 Interview Question</strong><br><em>"{question}"</em><br><br>
    <strong>Key Points:</strong> {key_points}</div>"""))

def success_box(content):
    """Display a success/best practice box."""
    display(HTML(f"""
    <div style="background:#E8F5E9;padding:15px;border-left:5px solid #4CAF50;
    margin:10px 0;border-radius:4px;font-family:Calibri,Arial,sans-serif;">
    <strong>✅ Best Practice</strong><br>{content}</div>"""))

def exercise_header(num, title, difficulty="⭐"):
    """Display a formatted exercise header."""
    display(HTML(f"""
    <div style="background:#E8EAF6;padding:15px;border-left:5px solid #0092D6;
    margin:15px 0;border-radius:4px;font-family:Calibri,Arial,sans-serif;">
    <strong>🏋️ Exercise {num}: {title}</strong> | Difficulty: {difficulty}</div>"""))

def draw_pipeline(steps, arrow="→"):
    """Draw a simple pipeline flow diagram."""
    flow = f" {arrow} ".join([f"[{s}]" for s in steps])
    display(HTML(f"""
    <div style="background:#F5F5F5;padding:20px;border-radius:8px;
    text-align:center;font-family:monospace;font-size:16px;margin:10px 0;">
    {flow}</div>"""))

print("✅ Helper functions loaded!")

## 🚀 Complete Working Example

Let's see the **entire ML workflow** from start to finish — loading data, engineering features, training a model, and making predictions. Run this cell first, then we'll break down every step.

In [ ]:
# ============================================================
# 🚀 COMPLETE WORKING EXAMPLE — Run me first!
# ============================================================
# This shows the full ML pipeline end-to-end.
# Don't worry about understanding every line yet!

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Step 1: Load data
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print("Step 1 ✅ Data loaded:", df.shape)

# Step 2: Separate features (X) from target (y)
X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]
print("Step 2 ✅ Features:", X.shape, "| Target:", y.shape)

# Step 3: Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Step 3 ✅ Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")

# Step 4: Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Note: only transform, don't fit!
print("Step 4 ✅ Features scaled")

# Step 5: Train model
model = LinearRegression()
model.fit(X_train_scaled, y_train)
print("Step 5 ✅ Model trained")

# Step 6: Evaluate
y_pred = model.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f"Step 6 ✅ RMSE: {rmse:.4f} | R² Score: {r2:.4f}")

print("\n🎉 Full pipeline complete!")

---
## 📖 Section 1: The Machine Learning Workflow

**What:** Machine learning follows a structured process — you don't just throw data at an algorithm and hope for the best!

**Why it matters:** In interviews and real projects, you'll be expected to follow a systematic approach. Skipping steps leads to models that fail in production.

**Analogy:** Think of ML like cooking a meal. You need to shop for ingredients (get data), prep them (clean & engineer features), follow a recipe (train a model), taste-test (evaluate), and then serve it (deploy). Skipping the prep step gives you a terrible meal!

In [ ]:
# Let's visualise the ML workflow
draw_pipeline([
    "Problem\nFraming",
    "Data\nCollection",
    "Data\nPrep",
    "Feature\nEngineering",
    "Model\nTraining",
    "Evaluation",
    "Deployment"
])

info_box(
    "The Golden Rule of ML",
    "Never use test data during training! The test set must remain unseen "
    "until final evaluation. This prevents <b>data leakage</b> — one of the "
    "most common mistakes beginners make."
)

---
## 📖 Section 2: Loading Data & Train-Test Split

**What:** Before training any model, we split our data into two parts — one for training and one for testing.

**Why it matters:** If we test on the same data we trained on, the model might just memorise the answers instead of learning patterns. This is called **overfitting**.

**Analogy:** It's like studying for an exam. If you only practise the exact questions that will appear on the test, you might score well — but you haven't really learned the subject. A good test uses questions you haven't seen before.

In [ ]:
# Let's load a beginner-friendly dataset
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
df = housing.frame

# Quick look at our data
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Understand what each column means
print("Feature descriptions:")
print("=" * 50)
for i, name in enumerate(housing.feature_names):
    print(f"  {name}")
print(f"\nTarget: MedHouseVal (Median house value in $100,000s)")
print(f"\nBasic statistics:")
df.describe().round(2)

In [ ]:
# Separate features (X) from target (y)
# X = the input data the model learns from
# y = the output we want the model to predict

X = df.drop("MedHouseVal", axis=1)  # Everything except the target
y = df["MedHouseVal"]               # Just the target column

print(f"Features (X): {X.shape}  — {X.shape[0]} rows, {X.shape[1]} columns")
print(f"Target  (y): {y.shape}   — {y.shape[0]} values")

info_box(
    "Features vs Target",
    "<b>Features (X)</b> = the information we give the model (inputs)<br>"
    "<b>Target (y)</b> = what we want the model to predict (output)"
)

In [ ]:
# Split data: 80% for training, 20% for testing
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% goes to the test set
    random_state=42      # Makes the split reproducible
)

print(f"Training set: {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test set:     {X_test.shape[0]} rows ({X_test.shape[0]/len(X)*100:.0f}%)")

warning_box(
    "Why random_state=42?",
    "Setting random_state ensures we get the <b>same split every time</b> we run "
    "the code. This makes our results reproducible. The number 42 is just a convention "
    "— you can use any integer."
)

---
## 📖 Section 3: Feature Scaling

**What:** Feature scaling transforms all features to a similar range so that no single feature dominates the model just because it has bigger numbers.

**Why it matters:** Many algorithms (like Linear Regression, KNN, SVM) are sensitive to the scale of features. A feature measured in millions will overpower one measured in single digits.

**Analogy:** Imagine comparing people's height (in cm, ~150–200) with their age (in years, ~0–100). Without scaling, the height values would unfairly dominate any distance calculations.

In [ ]:
# Look at the different scales of our features
print("Feature ranges BEFORE scaling:")
print("=" * 55)
for col in X_train.columns:
    min_val = X_train[col].min()
    max_val = X_train[col].max()
    print(f"  {col:15s}  min={min_val:10.2f}  max={max_val:10.2f}")

In [ ]:
# StandardScaler: transforms data to mean=0, std=1
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# IMPORTANT: fit on training data ONLY, then transform both
X_train_scaled = scaler.fit_transform(X_train)  # Learn + transform
X_test_scaled = scaler.transform(X_test)         # Only transform!

# Convert back to DataFrame for readability
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)

print("Feature ranges AFTER scaling:")
print("=" * 55)
for col in X_train_scaled_df.columns:
    min_val = X_train_scaled_df[col].min()
    max_val = X_train_scaled_df[col].max()
    print(f"  {col:15s}  min={min_val:10.2f}  max={max_val:10.2f}")

warning_box(
    "Common Mistake: Fitting on Test Data",
    "Always call <code>fit_transform()</code> on training data and "
    "<code>transform()</code> (without fit) on test data. Fitting on test data "
    "causes <b>data leakage</b> — your model peeks at the test set!"
)

In [ ]:
# Visualise the effect of scaling
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(X_train["MedInc"], bins=30, color="#0092D6", edgecolor="white")
axes[0].set_title("MedInc — Before Scaling", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Original Value")

axes[1].hist(X_train_scaled_df["MedInc"], bins=30, color="#8ACAE7", edgecolor="white")
axes[1].set_title("MedInc — After Scaling", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Scaled Value (mean=0, std=1)")

plt.tight_layout()
plt.show()

info_box(
    "Other Scalers You Should Know",
    "<b>MinMaxScaler</b> — scales to a range [0, 1]. Good for neural networks.<br>"
    "<b>RobustScaler</b> — uses median/IQR, less affected by outliers.<br>"
    "<b>StandardScaler</b> — uses mean/std. Most common choice."
)

---
## 📖 Section 4: Handling Missing Data

**What:** Real-world data almost always has missing values. We need to fill them in (impute) or remove them before training.

**Why it matters:** Most scikit-learn models cannot handle missing values directly. If you don't handle them, your code will crash!

In [ ]:
# Let's create a dataset with missing values to practise
df_missing = X_train.copy()

# Randomly insert some NaN values
np.random.seed(42)
for col in ["AveRooms", "AveBedrms", "Population"]:
    mask = np.random.random(len(df_missing)) < 0.1  # 10% missing
    df_missing.loc[mask, col] = np.nan

print("Missing values per column:")
print(df_missing.isnull().sum())
print(f"\nTotal missing: {df_missing.isnull().sum().sum()}")

In [ ]:
# SimpleImputer: fill missing values with a strategy
from sklearn.impute import SimpleImputer

# Strategy options: 'mean', 'median', 'most_frequent', 'constant'
imputer = SimpleImputer(strategy='median')  # Median is robust to outliers

# Fit on training data, transform both
df_imputed = pd.DataFrame(
    imputer.fit_transform(df_missing),
    columns=df_missing.columns,
    index=df_missing.index
)

print("Missing values AFTER imputation:")
print(df_imputed.isnull().sum())
print("\n✅ All missing values filled!")

success_box(
    "Use <b>median</b> for numerical features with outliers, "
    "<b>mean</b> when data is roughly symmetric, and "
    "<b>most_frequent</b> for categorical features."
)

---
## 📖 Section 5: Encoding Categorical Features

**What:** Machine learning models work with numbers, not text. Encoding converts text categories into numerical values.

**Why it matters:** If your dataset has columns like "colour" or "city", you must encode them before any model can use them.

In [ ]:
# Let's create a small example with categorical data
sample_data = pd.DataFrame({
    "city": ["London", "Paris", "London", "Berlin", "Paris", "Berlin", "London"],
    "size": ["small", "large", "medium", "large", "small", "medium", "large"],
    "price": [250, 400, 300, 380, 200, 320, 450]
})
print("Original data:")
sample_data

In [ ]:
# Method 1: One-Hot Encoding (for nominal categories — no natural order)
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' avoids redundancy
cities_encoded = ohe.fit_transform(sample_data[["city"]])
city_cols = ohe.get_feature_names_out(["city"])

print("One-Hot Encoded cities:")
pd.DataFrame(cities_encoded, columns=city_cols)

In [ ]:
# Method 2: Ordinal Encoding (for categories with a natural order)
from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder(categories=[["small", "medium", "large"]])
sample_data["size_encoded"] = oe.fit_transform(sample_data[["size"]])

print("Ordinal Encoded sizes:")
print(sample_data[["size", "size_encoded"]])

info_box(
    "One-Hot vs Ordinal Encoding",
    "<b>One-Hot</b>: Use for categories with NO natural order (city, colour).<br>"
    "<b>Ordinal</b>: Use for categories WITH a natural order (small/medium/large, low/high)."
)

---
## 📖 Section 6: Feature Selection

**What:** Not all features are equally useful. Feature selection helps us keep only the most informative ones.

**Why it matters:** Too many features can confuse the model (curse of dimensionality), slow down training, and lead to overfitting.

In [ ]:
# Method 1: Correlation analysis — find features most related to the target
correlations = df.corr()["MedHouseVal"].drop("MedHouseVal").sort_values(ascending=False)

print("Feature correlations with house price:")
print("=" * 45)
for feat, corr in correlations.items():
    bar = "█" * int(abs(corr) * 30)
    sign = "+" if corr > 0 else "-"
    print(f"  {feat:15s} {sign}{abs(corr):.3f}  {bar}")

In [ ]:
# Method 2: SelectKBest — statistical test to pick top features
from sklearn.feature_selection import SelectKBest, f_regression

selector = SelectKBest(score_func=f_regression, k=5)  # Keep top 5 features
X_selected = selector.fit_transform(X_train, y_train)

# Which features were selected?
selected_mask = selector.get_support()
selected_features = X_train.columns[selected_mask]

print(f"Selected {len(selected_features)} best features:")
for feat, score in zip(X_train.columns, selector.scores_):
    marker = "  ✅" if feat in selected_features.values else "  ❌"
    print(f"{marker} {feat:15s}  score = {score:.1f}")

interview_box(
    "How do you decide which features to include in your model?",
    "1. Start with domain knowledge — what features logically matter?<br>"
    "2. Check correlations with the target variable<br>"
    "3. Use statistical tests (SelectKBest, mutual information)<br>"
    "4. Try model-based selection (feature importances from tree models)<br>"
    "5. Consider removing highly correlated features (multicollinearity)"
)

---
## 🏋️ Exercises

In [ ]:
exercise_header(1, "Explore Train-Test Split", "⭐")

# Just run this cell and answer the questions below.
# We'll try different test_size values and see what happens.

for size in [0.1, 0.2, 0.3, 0.5]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=size, random_state=42)
    print(f"test_size={size:.1f}  →  Train: {len(Xtr):5d} rows  |  Test: {len(Xte):5d} rows")

# ❓ Questions:
# 1. What happens to the training set size as test_size increases?
# 2. Why might a very small test set (0.1) be a problem?
# 3. Why might a very large test set (0.5) be a problem?

In [ ]:
exercise_header(2, "Try Different Scalers", "⭐⭐")

# TODO: Compare StandardScaler and MinMaxScaler on the training data
from sklearn.preprocessing import MinMaxScaler

# StandardScaler (already done)
scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(X_train)

# MinMaxScaler — YOUR CODE HERE
scaler_mm = MinMaxScaler()
X_mm = # YOUR CODE HERE — use scaler_mm to fit_transform X_train

# Compare the results for the first feature (MedInc)
print("StandardScaler — MedInc:")
print(f"  Min: {X_std[:, 0].min():.3f}, Max: {X_std[:, 0].max():.3f}, Mean: {X_std[:, 0].mean():.3f}")
print("\nMinMaxScaler — MedInc:")
print(f"  Min: {X_mm[:, 0].min():.3f}, Max: {X_mm[:, 0].max():.3f}, Mean: {X_mm[:, 0].mean():.3f}")

In [ ]:
exercise_header(3, "Build a Simple Imputation Pipeline", "⭐⭐⭐")

# Challenge: Create a function that handles missing values AND scales the data.
# Use SimpleImputer (median strategy) followed by StandardScaler.

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

def preprocess_data(X_train_raw, X_test_raw):
    """
    Preprocess data by imputing missing values and scaling features.
    
    Parameters:
        X_train_raw: Training features (may contain NaN)
        X_test_raw: Test features (may contain NaN)
    
    Returns:
        X_train_clean: Cleaned and scaled training data
        X_test_clean: Cleaned and scaled test data
    """
    # Step 1: Impute missing values with median
    imputer = SimpleImputer(strategy='median')
    X_train_imputed = # YOUR CODE HERE — fit_transform on train
    X_test_imputed = # YOUR CODE HERE — only transform on test
    
    # Step 2: Scale features
    scaler = StandardScaler()
    X_train_clean = # YOUR CODE HERE — fit_transform on train
    X_test_clean = # YOUR CODE HERE — only transform on test
    
    return X_train_clean, X_test_clean

# Test with our data that has missing values
# X_train_clean, X_test_clean = preprocess_data(df_missing, X_test)
# assert X_train_clean.shape == df_missing.shape, "Shape should remain the same"
# assert not np.isnan(X_train_clean).any(), "No NaN values should remain"
# print("✅ All tests passed!")

---
## 📋 Solutions

<details>
<summary>Click to expand Exercise 2 solution</summary>

```python
scaler_mm = MinMaxScaler()
X_mm = scaler_mm.fit_transform(X_train)
```

</details>

<details>
<summary>Click to expand Exercise 3 solution</summary>

```python
def preprocess_data(X_train_raw, X_test_raw):
    imputer = SimpleImputer(strategy='median')
    X_train_imputed = imputer.fit_transform(X_train_raw)
    X_test_imputed = imputer.transform(X_test_raw)
    
    scaler = StandardScaler()
    X_train_clean = scaler.fit_transform(X_train_imputed)
    X_test_clean = scaler.transform(X_test_imputed)
    
    return X_train_clean, X_test_clean
```

</details>

---
## 🎯 Key Takeaways

1. **The ML workflow is systematic** — problem framing → data prep → feature engineering → training → evaluation → deployment
2. **Always split data before any preprocessing** — fit on training data, transform both
3. **Feature scaling** is essential for many algorithms — StandardScaler is the most common choice
4. **Handle missing data** before training — SimpleImputer with median is a safe default
5. **Encode categorical features** — One-Hot for unordered, Ordinal for ordered categories

## ✅ Self-Assessment Checklist
- [ ] I can explain why we split data into train and test sets
- [ ] I understand the difference between fit_transform and transform
- [ ] I know when to use StandardScaler vs MinMaxScaler
- [ ] I can handle missing values using SimpleImputer
- [ ] I can encode categorical features appropriately

## 📚 Next Steps
- **Next Lab:** Module 10 Lab 02 — Supervised Learning: Regression
- **Practice:** Try loading a different dataset from sklearn.datasets and apply the full preprocessing pipeline

---
*Prepared by Information Tech Consultants Ltd*